# Backblaze Drive Observations — Unified Bronze Ingestion 
Purpose: Consolidate the validated Q1 2016 and Q1 2018 Bronze tables into one period-aware Delta table.

**Target:**   backblaze_data.ing_bronze.drive_observations   
**Inputs:**  
- `backblaze_data.ing_bronze.drive_observations_2016_q1`
- `backblaze_data.ing_bronze.drive_observations_2018_q1`

**Conventions**  
- Preserve source values as strings.
- Align columns explicitly to the Q1 2018 source schema.
- Represent the ten columns introduced in Q1 2018 as NULL for Q1 2016.
- Preserve _source_file_path and _ingested_at; add _period.
- Replace one period at a time using Delta replaceWhere.
- Do not drop or modify the original period-specific tables.
- This notebook consolidates existing Bronze tables; it does not reread raw CSVs.

In [0]:
from pyspark.sql import functions as F

SOURCE_TABLE_2016 = "backblaze_data.ing_bronze.drive_observations_2016_q1"
SOURCE_TABLE_2018 = "backblaze_data.ing_bronze.drive_observations_2018_q1"
TARGET_TABLE = "backblaze_data.ing_bronze.drive_observations"

PERIOD_2016 = "2016_Q1"
PERIOD_2018 = "2018_Q1"

METADATA_COLUMNS = ["_source_file_path", "_ingested_at"]
REQUIRED_SOURCE_COLUMNS = {
    "date", "serial_number", "model", "capacity_bytes", "failure"
}

# These ten columns were introduced in the Q1 2018 source schema.
DRIFT_COLUMNS_2018 = [
    "smart_177_normalized",
    "smart_177_raw",
    "smart_179_normalized",
    "smart_179_raw",
    "smart_181_normalized",
    "smart_181_raw",
    "smart_182_normalized",
    "smart_182_raw",
    "smart_235_normalized",
    "smart_235_raw",
]

print(f"2016 source: {SOURCE_TABLE_2016}")
print(f"2018 source: {SOURCE_TABLE_2018}")
print(f"Unified target: {TARGET_TABLE}")

In [0]:
df_2016 = spark.table(SOURCE_TABLE_2016)
df_2018 = spark.table(SOURCE_TABLE_2018)

cols_2016 = set(df_2016.columns) - set(METADATA_COLUMNS)
cols_2018 = set(df_2018.columns) - set(METADATA_COLUMNS)

missing_required_2016 = REQUIRED_SOURCE_COLUMNS - cols_2016
missing_required_2018 = REQUIRED_SOURCE_COLUMNS - cols_2018
unexpected_2016 = cols_2016 - cols_2018
missing_drift_columns = set(DRIFT_COLUMNS_2018) - cols_2018

if missing_required_2016 or missing_required_2018:
    raise ValueError(
        f"Required columns missing. 2016={sorted(missing_required_2016)}, "
        f"2018={sorted(missing_required_2018)}"
    )

if unexpected_2016:
    raise ValueError(
        "Q1 2016 contains columns absent from Q1 2018 schema: "
        f"{sorted(unexpected_2016)}. Review schema drift before continuing."
    )

if missing_drift_columns:
    raise ValueError(
        f"Expected drift columns absent from Q1 2018: {sorted(missing_drift_columns)}"
    )

if not set(METADATA_COLUMNS).issubset(df_2016.columns) or not set(METADATA_COLUMNS).issubset(df_2018.columns):
    raise ValueError("Expected Bronze metadata columns are missing.")

source_columns = df_2018.columns
source_columns = [c for c in source_columns if c not in METADATA_COLUMNS]

print(f"Q1 2016 source columns: {len(cols_2016)}")
print(f"Q1 2018 source columns: {len(cols_2018)}")
print(f"Unified source columns: {len(source_columns)}")
print(f"Q1 2016 rows: {df_2016.count():,}")
print(f"Q1 2018 rows: {df_2018.count():,}")


In [0]:
def align_period(df, source_columns, period):
    # Existing source columns are selected in a fixed order.
    # Columns absent from the older source schema are intentional NULLs.
    expressions = []
    for column_name in source_columns:
        if column_name in df.columns:
            expressions.append(F.col(column_name).cast("string").alias(column_name))
        else:
            expressions.append(F.lit(None).cast("string").alias(column_name))

    expressions.extend([
        F.col("_source_file_path").cast("string").alias("_source_file_path"),
        F.col("_ingested_at").cast("timestamp").alias("_ingested_at"),
        F.lit(period).cast("string").alias("_period"),
    ])
    return df.select(*expressions)

unified_2016 = align_period(df_2016, source_columns, PERIOD_2016)
unified_2018 = align_period(df_2018, source_columns, PERIOD_2018)

expected_columns = source_columns + METADATA_COLUMNS + ["_period"]
if unified_2016.columns != expected_columns:
    raise ValueError("Q1 2016 aligned schema does not match unified contract.")
if unified_2018.columns != expected_columns:
    raise ValueError("Q1 2018 aligned schema does not match unified contract.")

print(f"Unified table column count: {len(expected_columns)}")
print("Expected: 105 source columns + 3 metadata columns = 108 total.")
print("Both period schemas aligned successfully.")


In [0]:
def write_period(df, period):
    table_exists = spark.catalog.tableExists(TARGET_TABLE)

    if not table_exists:
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .partitionBy("_period")
            .saveAsTable(TARGET_TABLE)
        )
        print(f"Created {TARGET_TABLE} with period {period}.")
    else:
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .option("replaceWhere", f"_period = '{period}'")
            .saveAsTable(TARGET_TABLE)
        )
        print(f"Replaced only period {period} in {TARGET_TABLE}.")

# Execute one period at a time. Start with Q1 2016.
write_period(unified_2016, PERIOD_2016)


In [0]:
# Validate the first load before loading the next period.
target_df = spark.table(TARGET_TABLE)

display(
    target_df.groupBy("_period")
    .agg(
        F.count("*").alias("rows"),
        F.countDistinct("_source_file_path").alias("source_files"),
        F.sum(F.when(F.col("_source_file_path").isNull(), 1).otherwise(0)).alias("null_source_paths"),
        F.sum(F.when(F.col("_ingested_at").isNull(), 1).otherwise(0)).alias("null_ingested_at"),
        F.sum(F.when(F.col("date").isNull(), 1).otherwise(0)).alias("null_dates"),
        F.sum(F.when(F.col("serial_number").isNull(), 1).otherwise(0)).alias("null_serial_numbers"),
    )
)

null_drift_2016 = unified_2016.select([
    F.sum(F.when(F.col(c).isNotNull(), 1).otherwise(0)).alias(c)
    for c in DRIFT_COLUMNS_2018
])
display(null_drift_2016)


In [0]:
# After Q1 2016 validation is acceptable, load Q1 2018.
write_period(unified_2018, PERIOD_2018)

In [0]:
# Final reconciliation: compare the unified table against the two source tables.
target_df = spark.table(TARGET_TABLE)

source_counts = {
    PERIOD_2016: df_2016.count(),
    PERIOD_2018: df_2018.count(),
}

actual_counts = {
    row["_period"]: row["count"]
    for row in target_df.groupBy("_period").count().collect()
}

for period, expected_count in source_counts.items():
    actual_count = actual_counts.get(period, 0)
    print(f"{period}: expected={expected_count:,}, actual={actual_count:,}")
    if actual_count != expected_count:
        raise ValueError(
            f"Row-count mismatch for {period}: expected {expected_count}, got {actual_count}"
        )

expected_periods = {PERIOD_2016, PERIOD_2018}
actual_periods = {r["_period"] for r in target_df.select("_period").distinct().collect()}
if actual_periods != expected_periods:
    raise ValueError(
        f"Unexpected period values. Expected {expected_periods}, got {actual_periods}"
    )

display(
    target_df.groupBy("_period")
    .agg(
        F.count("*").alias("total_rows"),
        F.countDistinct("_source_file_path").alias("source_files"),
        F.sum(F.when(F.col("_period").isNull(), 1).otherwise(0)).alias("null_periods"),
        F.sum(F.when(F.col("_source_file_path").isNull(), 1).otherwise(0)).alias("null_source_paths"),
        F.sum(F.when(F.col("_ingested_at").isNull(), 1).otherwise(0)).alias("null_ingested_at"),
        F.sum(F.when(F.col("date").isNull(), 1).otherwise(0)).alias("null_dates"),
        F.sum(F.when(F.col("serial_number").isNull(), 1).otherwise(0)).alias("null_serial_numbers"),
    )
)
print("Row-count reconciliation passed.")


In [0]:
# Validate expected schema-evolution NULLs in Q1 2016.
drift_nulls_2016 = target_df.filter(F.col("_period") == PERIOD_2016).agg(*[
    F.sum(F.when(F.col(c).isNotNull(), 1).otherwise(0)).alias(c)
    for c in DRIFT_COLUMNS_2018
])
display(drift_nulls_2016)

nonnull_drift_counts_2018 = target_df.filter(F.col("_period") == PERIOD_2018).agg(*[
    F.sum(F.when(F.col(c).isNotNull(), 1).otherwise(0)).alias(c)
    for c in DRIFT_COLUMNS_2018
])
display(nonnull_drift_counts_2018)

# Check the observation grain within each period. This is a diagnostic gate;
# investigate any duplicates rather than silently deduplicating Bronze.
duplicate_grain = (
    target_df.groupBy("_period", "date", "serial_number")
    .count()
    .filter(F.col("count") > 1)
)
duplicate_count = duplicate_grain.count()
print(f"Duplicate (_period, date, serial_number) groups: {duplicate_count:,}")
if duplicate_count:
    display(duplicate_grain.limit(20))
    raise ValueError("Duplicate observation grain found. Investigate before proceeding.")

print("Unified Bronze validation completed.")


In [0]:
from pyspark.sql import functions as F

before_2018 = (
    spark.table(TARGET_TABLE)
    .filter(F.col("_period") == PERIOD_2018)
    .count()
)

write_period(unified_2016, PERIOD_2016)

after_2018 = (
    spark.table(TARGET_TABLE)
    .filter(F.col("_period") == PERIOD_2018)
    .count()
)

print(f"Q1 2018 rows before: {before_2018:,}")
print(f"Q1 2018 rows after:  {after_2018:,}")

if before_2018 != after_2018:
    raise ValueError("Period-isolation check failed.")

print("Period-isolation check passed.")